# Exercise - Cross-Validation and the Train-Test Split

In this exercise, you will apply what you have learned about splitting the dataset into training and test sets, impute missing values and scale features the correct way to avoid data leakage, and perform k-fold cross-validation in order to get more reliable and representative estimates of the model's performance on unseen data.  

The dataset is a modified version of the ["Housing Prices Dataset" from Kaggle](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset).

In [ ]:
# DO NOT MODIFY - imports
import pandas as pd

## 1. Data Preparation

Other than a few missing values which were introduced intentionally for the purpose of this demo, the dataset is clean and free from duplicated rows and other issues. You do not need to write your own code in this section. However, please read this section and inspect the code thoroughly to understand how the dataset is being set up for the next step.

In [ ]:
# DO NOT MODIFY - Data loading and inspection
df = pd.read_csv("Housing_Modified_2.csv")
df.head()

In [ ]:
# DO NOT MODIFY - Check for missing values
df.isnull().sum()

We will impute the missing values in the `area` column.  
But first, run the cell below to convert the categorical "`yes`/`no`" columns to ones and zeros (integers).

In [ ]:
# DO NOT MODIFY - Data preparation
# Convert "yes" and "no" to 1 and 0
yes_no_columns = [
    "mainroad",
    "guestroom",
    "basement",
    "hotwaterheating",
    "airconditioning",
    "prefarea",
]
df[yes_no_columns] = df[yes_no_columns].map({"yes": 1, "no": 0}.get)
df.head()

In [ ]:
df.dtypes

Run the cell below to one-hot-encode the `furnishingstatus` column with the first resulting column (`furnished`) dropped to avoid multicollinearity.

In [ ]:
# DO NOT MODIFY - One-hot encoding `furnishingstatus`
df = pd.get_dummies(df, columns=["furnishingstatus"], drop_first=True)
df.dtypes

In [ ]:
df.head()

We are now ready to split the data, impute missing values and scale the features if need be.

## 2. Train-Test Split and Proper Imputation and Scaling

Create the feature set, the matrix `X`, consisting of all columns but `price`. Then create the target, the array `y`, comprised of the values in the `price` column.

In [ ]:
# FILL IN - Create feature set `X` and target `y`
X = df.drop(columns=["price"])
y = df["price"]

In [ ]:
X.head()

In [ ]:
y.head()

Split the data into training and testing sets using a 70/30 split. Shuffle the data while you split it, using a random seed of 52.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.model_selection import train_test_split

# FILL IN - Split the data into training and testing sets (70% train, 30% test) with a random state of 52
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=52
)

Impute missing values in the `area` column using the `SimpleImputer` class from Scikit-Learn. Use the `median` strategy.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.impute import SimpleImputer

# FILL IN - Fit the imputer on the training data, then transform the training AND test data using the fitted imputer
imputer = SimpleImputer(strategy="median")
X_train["area"] = imputer.fit_transform(X_train[["area"]])
X_test["area"] = imputer.transform(X_test[["area"]])

Below, we pick out columns of data that were originally numeric (and not just 0 or 1). Scale these features using a MinMaxScaler the correct way. - **HINT:** Only pass `X_train[numeric_columns]` and `X_test[numeric_columns]`, not all columns.

In [ ]:
#  DO NOT MODIFY - Features that were originally numeric (and not just 0 or 1)
numeric_columns = ["area", "bedrooms", "bathrooms", "stories", "parking"]

# DO NOT MODIFY - imports
from sklearn.preprocessing import MinMaxScaler

# FILL IN - Fit the MinMaxScaler on the training data, then transform the training AND test data using the fitted scaler
minMaxScaler = MinMaxScaler()
X_train[numeric_columns] = minMaxScaler.fit_transform(X_train[numeric_columns])
X_test[numeric_columns] = minMaxScaler.transform(X_test[numeric_columns])

Using `describe()`, verify that all values in both sets are between zero and one now.

In [ ]:
# FILL IN - `describe()` the training set
X_train.describe()

In [ ]:
X_test.describe()

## 3. K-Fold Cross-Validation

Train alinear regression model on the training set and output its *training* score (which, by default, is the R-squared for regression tasks). - **HINT:** Use the `score()` method of the fitted model.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.linear_model import LinearRegression

# FILL IN - Train a linear regression model and output its R-squared score on the training set
LinearRegressionModel = LinearRegression().fit(X_train, y_train)
RsquaredScore = LinearRegressionModel.score(X_train, y_train)

In [ ]:
print("R² square score: ", RsquaredScore)

Can we expect a similarly high score on unseen data? Before looking at the holdout (test) set, cross-validate the model using 5-fold CV and output the average score.

In [ ]:
# TBD: DO NOT MODIFY - imports
from sklearn.model_selection import cross_val_score

# FILL IN - Cross-validate the model using 5-fold CV and output the mean R-squared score
r2_scores = cross_val_score(LinearRegressionModel, X_train, y_train, cv=5, scoring="r2")
r2_scores_mean = r2_scores.mean()

print("Mean R-squared score: ", r2_scores_mean)

Finally, evaluate the trained model on the test set and output the test score (R-squared). Is it closer to the training score or the average CV score?

In [ ]:
# DO NOT MODIFY - imports
from sklearn.metrics import r2_score

# FILL IN - Evaluate the model on the test set and output its R-squared score
y_pred = LinearRegressionModel.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("R² score: ", r2)